# Фінальний проєкт. Реляційні бази даних

**Датасет:** [infectious_cases.csv](https://drive.google.com/file/d/1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl/view) — статистика інфекційних захворювань за країнами, регіонами та роками.

## 1. Завантаження даних

In [1]:
import sys, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

pkg = "pgserver" if sys.version_info < (3, 13) else "pixeltable-pgserver"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                       pkg, "jupysql", "psycopg2-binary", "sqlalchemy", "pandas", "gdown"])

import gdown
import pandas as pd
from sqlalchemy import create_engine, text

pgdata = Path("/tmp/fp_pg")
if pkg == "pgserver":
    import pgserver
    pg = pgserver.get_server(pgdata, cleanup_mode="stop")
else:
    import pixeltable_pgserver
    pg = pixeltable_pgserver.get_server(pgdata, postgres_version=16)

engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 0
%config SqlMagic.feedback = 0
%sql engine

PostgreSQL 16.14 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


In [2]:
csv_path = Path("data/infectious_cases.csv")
csv_path.parent.mkdir(parents=True, exist_ok=True)
if not csv_path.exists():
    gdown.download(id="1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl", output=str(csv_path), quiet=True)

with open(csv_path, encoding="utf-8") as file:
    print(file.readline().strip())
    print("CSV rows:", sum(1 for _ in file))

Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
CSV rows: 10521


### Схема `pandemic`

In [3]:
%%sql
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;

SELECT current_schema();

current_schema
pandemic


### Таблиця `infectious_cases`

Усі колонки `TEXT`, щоб імпортувати файл як є. Порожні значення зберігаються як `''` (`FORCE_NOT_NULL`), тож у таблиці видно, де в CSV немає числа.

In [4]:
%%sql
DROP TABLE IF EXISTS infectious_cases CASCADE;

CREATE TABLE infectious_cases (
    entity               TEXT,
    code                 TEXT,
    year                 TEXT,
    number_yaws          TEXT,
    polio_cases          TEXT,
    cases_guinea_worm    TEXT,
    number_rabies        TEXT,
    number_malaria       TEXT,
    number_hiv           TEXT,
    number_tuberculosis  TEXT,
    number_smallpox      TEXT,
    number_cholera_cases TEXT
);

SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_name = 'infectious_cases';

table_schema,table_name
pandemic,infectious_cases


### `COPY FROM STDIN`

In [5]:
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur, open(csv_path, encoding="utf-8", newline="") as file:
        cur.copy_expert(
            """
            COPY pandemic.infectious_cases (
                entity, code, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies,
                number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases
            )
            FROM STDIN WITH (
                FORMAT csv, HEADER true,
                FORCE_NOT_NULL (
                    entity, code, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies,
                    number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases
                )
            )
            """,
            file,
        )
        print("COPY rows:", cur.rowcount)
    raw_conn.commit()
finally:
    raw_conn.close()

COPY rows: 10521


### Огляд даних

In [6]:
%%sql
SELECT COUNT(*) AS n_rows FROM infectious_cases;

n_rows
10521


In [7]:
%%sql
SELECT *
FROM infectious_cases
LIMIT 10;

entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
Afghanistan,AFG,1980,,6160,0,,,,,,
Afghanistan,AFG,1981,,5859,0,,,,,,
Afghanistan,AFG,1982,,9730,0,,,,,,
Afghanistan,AFG,1983,,13937,0,,,,,,
Afghanistan,AFG,1984,,3864,0,,,,,,
Afghanistan,AFG,1985,,13867,0,,,,,,
Afghanistan,AFG,1986,,12901,0,,,,,,
Afghanistan,AFG,1987,,4396,0,,,,,,
Afghanistan,AFG,1988,,2149,0,,,,,,
Afghanistan,AFG,1989,,385,0,,,,,,


In [8]:
%%sql
SELECT COUNT(DISTINCT entity) AS n_entities,
       COUNT(DISTINCT entity) FILTER (WHERE code = '') AS n_entities_without_code,
       MIN(year::INTEGER) AS min_year,
       MAX(year::INTEGER) AS max_year,
       COUNT(*) FILTER (WHERE number_rabies <> '') AS rabies_filled,
       COUNT(*) FILTER (WHERE number_rabies = '') AS rabies_empty
FROM infectious_cases;

n_entities,n_entities_without_code,min_year,max_year,rabies_filled,rabies_empty
245,34,1920,2023,6780,3741


**Interpretation:** Імпортовано 10 521 рядок, стільки ж, скільки в CSV. У файлі 245 унікальних `entity`: країни та агреговані регіони. 34 з них (Africa, G20, World Bank і WHO-регіони) не мають `code`. Роки охоплюють 1920–2023, але `number_rabies` заповнено лише в 6 780 рядках, а 3 741 рядок порожній. Тому в аналізі ці порожні значення треба відфільтрувати.

## 2. Нормалізація до 3NF

Вихідна таблиця вже в 1NF: кожна клітинка містить одне значення. Ключ рядка — `(entity, year)`. Але `code` залежить лише від `entity`, тобто від частини ключа, а не від усього ключа. Це порушення 2NF, через яке пара `entity` + `code` повторюється в кожному році.

Розбиваємо дані на дві таблиці:

- `entities` — унікальні пари `entity` + `code` із сурогатним `id`;
- `infectious_cases_normalized` — показники за `(entity_id, year)`.

У другій таблиці всі показники залежать від повного ключа `(entity_id, year)`, і між неключовими колонками немає залежностей, тож обидві таблиці в 3NF. Порожні рядки стають `NULL`, а колонки отримують числові типи: `INTEGER` для цілих лічильників і `NUMERIC` для модельованих оцінок із дробовою частиною (rabies, malaria, HIV, tuberculosis).

### Перевірка залежності `entity → code`

In [9]:
%%sql
SELECT COUNT(*) AS entities_with_several_codes
FROM (
    SELECT entity
    FROM infectious_cases
    GROUP BY entity
    HAVING COUNT(DISTINCT code) > 1
) AS t;

entities_with_several_codes
0


### Таблиця `entities`

In [10]:
%%sql
DROP TABLE IF EXISTS infectious_cases_normalized;
DROP TABLE IF EXISTS entities;

CREATE TABLE entities (
    id     INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity TEXT NOT NULL,
    code   TEXT,
    CONSTRAINT uq_entities_entity_code UNIQUE NULLS NOT DISTINCT (entity, code)
);

INSERT INTO entities (entity, code)
SELECT DISTINCT entity, NULLIF(code, '')
FROM infectious_cases
ORDER BY entity;

SELECT COUNT(*) AS n_entities,
       COUNT(*) FILTER (WHERE code IS NULL) AS without_code
FROM entities;

n_entities,without_code
245,34


In [11]:
%%sql
SELECT * FROM entities ORDER BY id LIMIT 10;

id,entity,code
1,Afghanistan,AFG
2,Africa,None
3,African Region (WHO),None
4,Albania,ALB
5,Algeria,DZA
6,American Samoa,ASM
7,Americas,None
8,Andorra,AND
9,Angola,AGO
10,Antigua and Barbuda,ATG


### Таблиця `infectious_cases_normalized`

In [12]:
%%sql
CREATE TABLE infectious_cases_normalized (
    entity_id            INTEGER NOT NULL REFERENCES entities (id),
    year                 SMALLINT NOT NULL CHECK (year BETWEEN 1900 AND 2100),
    number_yaws          INTEGER,
    polio_cases          INTEGER,
    cases_guinea_worm    INTEGER,
    number_rabies        NUMERIC,
    number_malaria       NUMERIC,
    number_hiv           NUMERIC,
    number_tuberculosis  NUMERIC,
    number_smallpox      INTEGER,
    number_cholera_cases INTEGER,
    PRIMARY KEY (entity_id, year)
);

INSERT INTO infectious_cases_normalized
SELECT e.id,
       ic.year::SMALLINT,
       NULLIF(ic.number_yaws, '')::INTEGER,
       NULLIF(ic.polio_cases, '')::INTEGER,
       NULLIF(ic.cases_guinea_worm, '')::INTEGER,
       NULLIF(ic.number_rabies, '')::NUMERIC,
       NULLIF(ic.number_malaria, '')::NUMERIC,
       NULLIF(ic.number_hiv, '')::NUMERIC,
       NULLIF(ic.number_tuberculosis, '')::NUMERIC,
       NULLIF(ic.number_smallpox, '')::INTEGER,
       NULLIF(ic.number_cholera_cases, '')::INTEGER
FROM infectious_cases AS ic
JOIN entities AS e
  ON e.entity = ic.entity
 AND e.code IS NOT DISTINCT FROM NULLIF(ic.code, '');

SELECT COUNT(*) AS n_rows FROM infectious_cases_normalized;

n_rows
10521


In [13]:
%%sql
SELECT * FROM infectious_cases_normalized ORDER BY entity_id, year LIMIT 10;

entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
1,1960,None,None,None,None,None,None,None,None,887
1,1965,None,None,None,None,None,None,None,None,218
1,1980,None,6160,0,None,None,None,None,None,None
1,1981,None,5859,0,None,None,None,None,None,None
1,1982,None,9730,0,None,None,None,None,None,None
1,1983,None,13937,0,None,None,None,None,None,None
1,1984,None,3864,0,None,None,None,None,None,None
1,1985,None,13867,0,None,None,None,None,None,None
1,1986,None,12901,0,None,None,None,None,None,None
1,1987,None,4396,0,None,None,None,None,None,None


### Перевірка: з нормалізованих таблиць відновлюється оригінал

In [14]:
%%sql
WITH restored AS (
    SELECT e.entity, COALESCE(e.code, '') AS code, n.year, n.number_rabies
    FROM infectious_cases_normalized AS n
    JOIN entities AS e ON e.id = n.entity_id
),
original AS (
    SELECT entity, code, year::SMALLINT AS year, NULLIF(number_rabies, '')::NUMERIC AS number_rabies
    FROM infectious_cases
)
SELECT (SELECT COUNT(*) FROM (SELECT * FROM original EXCEPT SELECT * FROM restored) AS a) AS missing_in_normalized,
       (SELECT COUNT(*) FROM (SELECT * FROM restored EXCEPT SELECT * FROM original) AS b) AS extra_in_normalized;

missing_in_normalized,extra_in_normalized
0,0


**Interpretation:** Кожна `entity` має рівно один `code`, тому залежність `entity → code` виконується, і винести її в окрему таблицю коректно. У `entities` 245 рядків, з них 34 без коду. `UNIQUE NULLS NOT DISTINCT` не дає продублювати регіон із `code = NULL`. У `infectious_cases_normalized` 10 521 рядок, як в оригіналі. Обидва `EXCEPT` повертають 0: з нормалізованих таблиць через `JOIN` відновлюються ті самі `entity`, `code`, `year` і `number_rabies`.

## 3. Аналіз `number_rabies`

Рядки без значення відкидаємо в `WHERE`: інакше країни, для яких немає жодного значення, отримали б `AVG = NULL`, а в PostgreSQL `NULL` при `ORDER BY ... DESC` іде першим.

In [15]:
%%sql
SELECT e.id,
       e.entity,
       e.code,
       ROUND(AVG(n.number_rabies), 4) AS avg_rabies,
       MIN(n.number_rabies) AS min_rabies,
       MAX(n.number_rabies) AS max_rabies,
       SUM(n.number_rabies) AS sum_rabies
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
GROUP BY e.id, e.entity, e.code
ORDER BY avg_rabies DESC
LIMIT 10;

id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
241,World,OWID_WRL,20192.3704,14075.508,24744.658,605771.111
121,Lower Middle Income (WB),None,15193.9594,10202.528,19182.795,455818.7815
202,South Asia (WB),None,11729.8888,7271.2754,15361.878,351896.6626
206,South-East Asia Region (WHO),None,11424.3269,6806.0127,15641.958,342729.8068
77,G20,None,10189.0460,6339.0796,13164.881,305671.3801
97,India,IND,8599.1733,5425.8726,11121.142,257975.1984
209,Sub-Saharan Africa (WB),None,4748.2681,4356.8184,5017.977,142448.0427
3,African Region (WHO),None,4649.7024,4283.073,4917.784,139491.0713
120,Low Income (WB),None,3568.1055,3014.617,3873.533,107043.1635
58,East Asia & Pacific (WB),None,3427.4611,2009.9395,4591.0107,102823.8340


**Interpretation:** Для кожної з 10 `entity` агрегуються 30 років (1990–2019), бо `number_rabies` заповнено тільки за ці роки. Найвище середнє в `World` — 20 192 випадки на рік. Далі йдуть агрегати Lower Middle Income, South Asia, South-East Asia Region (WHO) і G20. Серед усіх 10 рядків лише одна країна — India (8 599). Причина в тому, що датасет змішує країни й регіони. Для порівняння саме країн варто додати фільтр `e.code IS NOT NULL AND e.code NOT LIKE 'OWID%'`.

## 4. Різниця в роках

`MAKE_DATE(year, 1, 1)` — перше січня року, `CURRENT_DATE` — поточна дата, `AGE` повертає інтервал між ними, а `EXTRACT(YEAR FROM ...)` бере з нього кількість повних років.

In [16]:
%%sql
SELECT e.entity,
       n.year,
       MAKE_DATE(n.year, 1, 1) AS year_start,
       CURRENT_DATE AS today,
       EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INTEGER AS years_diff
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
ORDER BY e.entity, n.year
LIMIT 10;

entity,year,year_start,today,years_diff
Afghanistan,1960,1960-01-01,2026-10-19,66
Afghanistan,1965,1965-01-01,2026-10-19,61
Afghanistan,1980,1980-01-01,2026-10-19,46
Afghanistan,1981,1981-01-01,2026-10-19,45
Afghanistan,1982,1982-01-01,2026-10-19,44
Afghanistan,1983,1983-01-01,2026-10-19,43
Afghanistan,1984,1984-01-01,2026-10-19,42
Afghanistan,1985,1985-01-01,2026-10-19,41
Afghanistan,1986,1986-01-01,2026-10-19,40
Afghanistan,1987,1987-01-01,2026-10-19,39


**Interpretation:** `year_start` — перше січня відповідного року, `today` — дата запуску notebook. Оскільки поточна дата вже після 1 січня, `years_diff` дорівнює поточний рік мінус `year`: наприклад, для 1980 це 46. Результат залежить від дати запуску, тож при іншому запуску значення в `today` і `years_diff` будуть іншими.

## 5. Власні функції

### Функція різниці в роках

In [17]:
%%sql
CREATE OR REPLACE FUNCTION fn_years_since(p_year INTEGER)
RETURNS INTEGER
LANGUAGE sql
STABLE
AS $$
    SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INTEGER;
$$;

SELECT 1996 AS year,
       MAKE_DATE(1996, 1, 1) AS year_start,
       CURRENT_DATE AS today,
       fn_years_since(1996) AS years_diff;

year,year_start,today,years_diff
1996,1996-01-01,2026-10-19,30


In [18]:
%%sql
SELECT e.entity,
       n.year,
       fn_years_since(n.year) AS years_diff
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
ORDER BY e.entity, n.year
LIMIT 10;

entity,year,years_diff
Afghanistan,1960,66
Afghanistan,1965,61
Afghanistan,1980,46
Afghanistan,1981,45
Afghanistan,1982,44
Afghanistan,1983,43
Afghanistan,1984,42
Afghanistan,1985,41
Afghanistan,1986,40
Afghanistan,1987,39


In [19]:
%%sql
SELECT COUNT(*) AS n_rows,
       COUNT(*) FILTER (
           WHERE fn_years_since(year) <> EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER
       ) AS mismatches_with_task_4
FROM infectious_cases_normalized;

n_rows,mismatches_with_task_4
10521,0


### Функція середньої кількості захворювань за період

In [20]:
%%sql
CREATE OR REPLACE FUNCTION fn_cases_per_period(p_cases_per_year NUMERIC, p_divisor NUMERIC)
RETURNS NUMERIC
LANGUAGE sql
IMMUTABLE
AS $$
    SELECT p_cases_per_year / NULLIF(p_divisor, 0);
$$;

SELECT fn_cases_per_period(1200, 12) AS per_month,
       fn_cases_per_period(1200, 4) AS per_quarter,
       fn_cases_per_period(1200, 2) AS per_half_year,
       fn_cases_per_period(1200, 0) AS divisor_zero;

per_month,per_quarter,per_half_year,divisor_zero
100.0000000000000000,300.0000000000000000,600.0000000000000000,None


Беремо лише рядки, де `number_rabies` в оригінальній таблиці є числом:

In [21]:
%%sql
SELECT COUNT(*) FILTER (WHERE number_rabies ~ '^[0-9]+(\.[0-9]+)?$') AS numeric_rows,
       COUNT(*) FILTER (WHERE number_rabies !~ '^[0-9]+(\.[0-9]+)?$') AS non_numeric_rows
FROM infectious_cases;

numeric_rows,non_numeric_rows
6780,3741


In [22]:
%%sql
SELECT entity,
       year,
       number_rabies::NUMERIC AS rabies_per_year,
       ROUND(fn_cases_per_period(number_rabies::NUMERIC, 12), 4) AS rabies_per_month,
       ROUND(fn_cases_per_period(number_rabies::NUMERIC, 4), 4) AS rabies_per_quarter,
       ROUND(fn_cases_per_period(number_rabies::NUMERIC, 2), 4) AS rabies_per_half_year
FROM infectious_cases
WHERE number_rabies ~ '^[0-9]+(\.[0-9]+)?$'
ORDER BY entity, year
LIMIT 10;

entity,year,rabies_per_year,rabies_per_month,rabies_per_quarter,rabies_per_half_year
Afghanistan,1990,2.145772,0.1788,0.5364,1.0729
Afghanistan,1991,2.3209748,0.1934,0.5802,1.1605
Afghanistan,1992,2.6180408,0.2182,0.6545,1.3090
Afghanistan,1993,2.8531172,0.2378,0.7133,1.4266
Afghanistan,1994,2.9899871,0.2492,0.7475,1.4950
Afghanistan,1995,3.126361,0.2605,0.7816,1.5632
Afghanistan,1996,3.281437,0.2735,0.8204,1.6407
Afghanistan,1997,3.446122,0.2872,0.8615,1.7231
Afghanistan,1998,3.5570247,0.2964,0.8893,1.7785
Afghanistan,1999,3.607037,0.3006,0.9018,1.8035


**Interpretation:** `fn_years_since(1996)` повертає 30, як у прикладі з умови, а на всіх 10 521 рядках дає той самий результат, що й вбудовані функції із завдання 4 (`mismatches_with_task_4 = 0`). `fn_cases_per_period` ділить річну кількість на дільник: з 1 200 виходить 100 на місяць, 300 на квартал і 600 на півріччя, а для дільника 0 повертається `NULL` замість помилки. Регулярний вираз відбирає 6 780 числових значень `number_rabies` і відкидає 3 741 порожнє. Для Afghanistan 1990 (2.145772 випадку на рік) функція дає 0.1788 на місяць, 0.5364 на квартал і 1.0729 на півріччя.